In [1]:
import pandas as pd

# df = pd.read_csv("/home/saliherdemk/osu-dataset/encoded.csv", chunksize = 100_000)
# df = next(iter(df))
df = pd.read_csv("/home/saliherdemk/osu-dataset/encoded.csv")
df

,beatmap_id,chunk,tokenized
0,1000488-0,0,"<beatmap_start>,<hit_object_start>,type_circle..."
1,1000488-0,1,"<hit_object_start>,type_slider,<start_delta_ti..."
2,1000488-0,2,"<hit_object_start>,type_slider,<start_delta_ti..."
3,1000488-0,3,"<hit_object_start>,type_circle,<start_delta_ti..."
4,1000488-0,4,"<hit_object_start>,type_slider,<start_delta_ti..."
...,...,...,...
1433788,999421-1,15,"<hit_object_start>,type_slider,<start_delta_ti..."
1433789,999421-1,16,"<hit_object_start>,type_slider,<start_delta_ti..."
1433790,999421-1,17,"<hit_object_start>,type_slider,<start_delta_ti..."
1433791,999421-1,18,"<hit_object_start>,type_circle,<start_delta_ti..."


In [6]:
df["beatmap"] = df["beatmap_id"].str.split("-").str[0]

In [15]:
a = list(df["beatmap"].unique())
with open("/home/saliherdemk/osu-dataset/ids.txt", 'w') as f:
    f.write("\n".join(map(str, a)))

In [2]:
def last_hit_object_index(tokenized_str):
    tokens = tokenized_str.split(',')
    try:
        last_idx= len(tokens) - 1 - tokens[::-1].index('<hit_object_start>')
        return ','.join(tokens[last_idx:])
    except ValueError:
        return ''

df['last_hit_object'] = df['tokenized'].apply(last_hit_object_index)

In [3]:
df['prev_last_hit_object'] = df.groupby('beatmap_id')['last_hit_object'].shift(1)

In [4]:
df[(df["prev_last_hit_object"].isna()) & (df["chunk"] != 0)]

,beatmap_id,chunk,tokenized,last_hit_object,prev_last_hit_object


In [5]:
prev = df['prev_last_hit_object'].fillna("")

In [6]:
df['tokenized_with_overlap'] = (prev + "," + df['tokenized']).str.strip(",")

In [7]:
df["count"] = df["tokenized_with_overlap"].str.count(",") + 1

In [11]:
df["count_vanilla"] = df["tokenized"].str.count(",") + 1

In [12]:
df["count"].max(), df["count_vanilla"].max()

(np.int64(1963), np.int64(1950))

In [13]:
grouped = df.groupby("beatmap_id")

In [ ]:
samples = {}
for b_id, data in grouped:
    chunks = []
    for _, row in data.sort_values("chunk").iterrows():
        tokens = row["tokenized_with_overlap"].split(",")
        chunks.append(tokens)
    samples[b_id] = chunks
    break
samples

In [ ]:
a = df[(df["beatmap_id"] == "2044490-1") & (df["chunk"] == 14)]
a["last_hit_object"].iloc[0]

In [ ]:
f = df[(df["beatmap_id"] == "2044490-1") & (df["chunk"] == 15)]

f["tokenized_with_overlap"].iloc[0]

In [ ]:
f["tokenized"].iloc[0]

In [ ]:
df[(df['last_hit_object'] == -1) & (df['count'] != 2)]

# DataLoader

In [5]:
from torch.utils.data import Dataset
import os
import json

class BeatmapDataset(Dataset):
    
    def __init__(self, df, mel_folder, tok2id_path):
        self.mel_folder = mel_folder
        self.beatmap_info = []
        with open(tok2id_path, 'r') as f:
            self.tok2id_dict = json.load(f)
        
        df['last_hit_object'] = df['tokenized'].apply(self._last_hit_object)
        df['prev_last_hit_object'] = df.groupby('beatmap_id')['last_hit_object'].shift(1)
        prev = df["prev_last_hit_object"].fillna("")
        df["tokenized_with_overlap"] = (prev + "," + df["tokenized"]).str.strip(",")
        
        for b_id, data in df.groupby("beatmap_id"):
            sorted_data = data.sort_values("chunk")
            
            chunk_info = []
            for _, row in sorted_data.iterrows():
                mel_path = os.path.join(self.mel_folder, f"{row['beatmap_id'].split('-')[0]}_{row['chunk']}.npy")
                chunk_info.append({
                    'mel_path': mel_path,
                    'tokenized': row["tokenized_with_overlap"]
                })
            
            self.beatmap_info.append({
                'beatmap_id': b_id,
                'chunks': chunk_info
            })
        
        del df
    
    def _last_hit_object(self, tokenized_str):
        tokens = tokenized_str.split(',')
        try:
            last_idx = len(tokens) - 1 - tokens[::-1].index('<hit_object_start>')
            return ','.join(tokens[last_idx:])
        except ValueError:
            return ''
    
    def __len__(self):
        return len(self.beatmap_info)
    
    def __getitem__(self, idx):
        beatmap = self.beatmap_info[idx]
        
        mel_list = []
        chunk_tokens = []
        
        for chunk_info in beatmap['chunks']:
            mel = np.load(chunk_info['mel_path'])
            mel_list.append(mel)
            
            tokens = chunk_info['tokenized'].split(",")
            token_ids = [self.tok2id_dict[tok] for tok in tokens if len(tok)]
            chunk_tokens.append(token_ids)
        
        return mel_list, chunk_tokens

In [6]:
from torch.utils.data import DataLoader
import torch
from torch.nn.utils.rnn import pad_sequence

pad_token = 4479

def collate_fn(batch):
    all_mels = []
    all_tokens = []
    all_masks = []

    for mel_list, chunk_tokens in batch:
        mel_tensor = torch.stack([torch.tensor(m, dtype=torch.float32) for m in mel_list], dim=0)
        all_mels.append(mel_tensor)

        chunk_tensors = [torch.tensor(seq, dtype=torch.long) for seq in chunk_tokens]
        chunk_padded = pad_sequence(chunk_tensors, batch_first=True, padding_value=pad_token)
        
        chunk_mask = (chunk_padded != pad_token).long()
        
        all_tokens.append(chunk_padded)
        all_masks.append(chunk_mask)
       

    mel_batch = torch.stack(all_mels)
    token_batch = torch.stack(all_tokens)
    mask = torch.stack(all_masks)

    return mel_batch, token_batch, mask


dataset = BeatmapDataset(df, mel_folder="/home/saliherdemk/osu-dataset/mels/", tok2id_path="/home/saliherdemk/projects/osu-dataset-generator/Tokenizer/vocab/token2id.json")

dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn
)
import psutil
process = psutil.Process()
print(f"Memory usage: {process.memory_info().rss / 1024 / 1024:.1f} MB")

Memory usage: 2598.4 MB


In [11]:
import numpy as np
for mel_batch, token_batch, mask in dataloader:
    print(mel_batch.shape)
    print(token_batch.shape)
    print(mask.shape)
    break

torch.Size([1, 21, 512, 128])
torch.Size([1, 21, 455])
torch.Size([1, 21, 455])


# Model

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EncoderLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=2, bidirectional=False, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, x, hidden=None):
        outputs, hidden = self.lstm(x, hidden)
        return outputs, hidden

class DecoderLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, dropout=0.1, pad_idx=pad_token):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids, hidden=None):
        embedded = self.embedding(token_ids)

        outputs, hidden = self.lstm(embedded, hidden)

        logits = self.fc_out(outputs)

        return logits, hidden


In [10]:
n_mels = 128
hidden_dim = 512

encoder = EncoderLSTM(input_dim=n_mels, hidden_dim=hidden_dim, num_layers=2, bidirectional=False)

for mel_batch, token_batch, mask in dataloader:
    encoder_outputs, hidden = encoder(mel_batch[0])
    print(encoder_outputs.shape)  # [1, 512, 512] since unidirectional
    print(hidden[0].shape)        # h_n: [num_layers, batch, hidden_dim]
    print(hidden[1].shape)        # c_n: [num_layers, batch, hidden_dim]
    break

torch.Size([21, 512, 512])
torch.Size([2, 21, 512])
torch.Size([2, 21, 512])


# Pretrained Model

In [15]:
class MelAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.freq_converter = nn.Linear(128, 80)
        
        self.time_converter = nn.Sequential(
            nn.Conv1d(80, 160, kernel_size=3, padding=1),
            nn.ReLU(), 
            nn.Conv1d(160, 80, kernel_size=3, padding=1),
            nn.Upsample(size=1200, mode='linear', align_corners=False)
        )
        
    def forward(self, mel):
        # Input: (batch, chunks, 512, 128) 
        batch, chunks, time, freq = mel.shape
        
        mel = mel.view(-1, time, freq)  # (batch*chunks, 512, 128)
        mel = self.freq_converter(mel)  # (batch*chunks, 512, 80)
        mel = mel.transpose(1, 2)  # (batch*chunks, 80, 512)
        mel = self.time_converter(mel)  # (batch*chunks, 80, 1200)
        
        return mel.view(batch, chunks, 80, 1200)

In [26]:
from transformers import WhisperModel, T5ForConditionalGeneration
import torch.nn as nn

class AudioToBeatmapModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.mel_adapter = MelAdapter()
        
        self.whisper_encoder = WhisperModel.from_pretrained("openai/whisper-base").encoder
        for param in self.whisper_encoder.parameters():
            param.requires_grad = False
            
        # self.t5_decoder = T5ForConditionalGeneration.from_pretrained("t5-base")
        # self.t5_decoder.resize_token_embeddings(vocab_size)
        
        # self.connection = nn.Linear(512, 768)
        
    def forward(self, mel_batch, token_batch=None):
        adapted_mels = self.mel_adapter(mel_batch)  # (1, 21, 80, 1200)
        
        batch_size, num_chunks = adapted_mels.shape[:2]
        
        # Process each chunk through Whisper encoder
        chunk_features = []
        for i in range(num_chunks):
            chunk_mel = adapted_mels[:, i]  # (1, 80, 1200)
            features = self.whisper_encoder(chunk_mel).last_hidden_state
            chunk_features.append(features)
        
        # Concatenate all chunk features  
        all_features = torch.cat(chunk_features, dim=1)
        return all_features
        # mapped_features = self.connection(all_features)

In [ ]:
import numpy as np

model = AudioToBeatmapModel()
for mel_batch, token_batch, mask in dataloader:
    with torch.no_grad():
        output = model(mel_batch, token_batch)
        print(output.shape)
    break

In [28]:
import numpy as np
for mel_batch, token_batch, mask in dataloader:
    print(mel_batch.shape)
    print(meladopter(mel_batch).shape)
    break

torch.Size([1, 21, 512, 128])
torch.Size([1, 21, 80, 1200])


# MERT

In [30]:
import torch
import torch.nn as nn

class LearnableMelProjection(nn.Module):
    def __init__(self, in_bins=128, out_bins=80):
        super().__init__()
        self.proj = nn.Linear(in_bins, out_bins)
    
    def forward(self, mel_spec):

        if mel_spec.dim() == 2:
            mel_spec = mel_spec.unsqueeze(0)  # [1, time, in_bins]
        
        out = self.proj(mel_spec)  # [batch, time, out_bins]
        return out


In [33]:
for mel_batch, token_batch, mask in dataloader:
    mel_spec = torch.tensor(mel_batch, dtype=torch.float32)
    
    mel_proj = LearnableMelProjection(in_bins=128, out_bins=80)
    
    mel_for_mert = mel_proj(mel_spec)  # [1, time, 80]
    batch, chunks, time, feat = mel_for_mert.shape
    mel_flat = mel_for_mert.view(batch, chunks * time, feat)
    print(mel_batch.shape)
    print(mel_flat.shape)
    
    break
    



torch.Size([1, 21, 512, 128])
torch.Size([1, 10752, 80])


/tmp/ipykernel_106916/3257919716.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mel_spec = torch.tensor(mel_batch, dtype=torch.float32)


In [ ]:
from transformers import AutoModel
import torch

mert_model = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
mert_model.eval()


In [36]:
from transformers import AutoModel

mert_model.eval()
with torch.no_grad():
    outputs = mert_model(inputs_embeds=mel_flat, output_hidden_states=True)

all_layer_hidden_states = torch.stack(outputs.hidden_states).squeeze(1)  # [num_layers, seq_len, hidden_dim]
print(all_layer_hidden_states.shape)
# # Option 1: average across layers
# encoder_embeddings = all_layer_hidden_states.mean(0)  # [seq_len, hidden_dim]

# # Option 2: learnable weighted average across layers
# import torch.nn as nn
# aggregator = nn.Conv1d(in_channels=all_layer_hidden_states.shape[0], out_channels=1, kernel_size=1)
# weighted_embeddings = aggregator(all_layer_hidden_states.permute(1, 2, 0).unsqueeze(0)).squeeze()  # [seq_len, hidden_dim]


TypeError: MERTModel.forward() got an unexpected keyword argument 'inputs_embeds'